In [38]:
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from roc_helper import plot_roc
from sklearn.metrics import roc_curve, auc

In [39]:
training_data = pd.read_csv("./data/train.csv")
test_data = pd.read_csv("./data/test.csv")

In [40]:
X_train = training_data.copy()
y_train = X_train['Transported'].copy()
X_submission_test = test_data.copy()


## Data handling

Extract passenger group from PassengerId.

In [41]:
# Data handling aligned with main branch style: straightforward cleanup and consistent train->test transforms
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def preprocess_spaceship(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    for df in (train_df, test_df):
        if 'PassengerId' in df.columns:
            parts = df['PassengerId'].astype(str).str.split('_', expand=True)
            df['Group_num'] = parts[0]
            df['Group_ID'] = parts[1]
            df['Group_size'] = df.groupby('Group_num')['Group_num'].transform('count')

        if 'Cabin' in df.columns:
            cabin_parts = df['Cabin'].astype(str).str.split('/', expand=True)
            df['Deck'] = cabin_parts[0].replace('nan', np.nan)
            df['Cabin_num'] = pd.to_numeric(cabin_parts[1], errors='coerce')
            df['Side'] = cabin_parts[2].replace('nan', np.nan)

        for c in spend_cols:
            if c in df.columns:
                df[c] = df[c].fillna(0)
        present_spend_cols = [c for c in spend_cols if c in df.columns]
        if present_spend_cols:
            df['TotalSpent'] = df[present_spend_cols].sum(axis=1)

        if 'Name' in df.columns:
            df['Surname'] = df['Name'].astype(str).str.split().str[-1]

    bool_cols = [c for c in ['CryoSleep', 'VIP'] if c in train_df.columns]
    for col in bool_cols:
        train_df[col] = train_df[col].map({True: 1, False: 0})
        if col in test_df.columns:
            test_df[col] = test_df[col].map({True: 1, False: 0})

    numeric_cols = train_df.select_dtypes(include=['number']).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != 'Transported']
    fill_values_num = train_df[numeric_cols].median(numeric_only=True)
    train_df[numeric_cols] = train_df[numeric_cols].fillna(fill_values_num)
    overlap_num = [c for c in numeric_cols if c in test_df.columns]
    test_df[overlap_num] = test_df[overlap_num].fillna(fill_values_num[overlap_num])

    categorical_cols = [
        c for c in train_df.columns
        if c not in numeric_cols + ['Transported']
    ]
    fill_values_cat = {}
    for col in categorical_cols:
        mode = train_df[col].dropna().mode()
        fill_values_cat[col] = mode.iloc[0] if not mode.empty else 'Unknown'
        train_df[col] = train_df[col].fillna(fill_values_cat[col])
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(fill_values_cat[col])

    return train_df, test_df

X_train, X_submission_test = preprocess_spaceship(X_train, X_submission_test)
X_train.head()


,Group_num,Group_ID,Group_size
0,0001,01,1
1,0002,01,1
2,0003,01,2
3,0003,02,2
4,0004,01,1


In [42]:
# Data handling is performed in cell 5.


,Group_num,Group_ID,Group_size
0,0013,01,1
1,0018,01,1
2,0019,01,1
3,0021,01,1
4,0023,01,1


Ensure GroupKFold cross validation using passenger groupp as grouping variable

In [43]:
from sklearn.model_selection import GroupKFold
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)
print('Prepared GroupKFold with', n_splits, 'folds')

Prepared GroupKFold with 5 folds


Use 5 folds

Track mean and std CV accuracy

## Feature engineering

### Cabin
Split into Deck, CabinNumber, Side
Create Deck frequency encoding (out of fold)
Create relative cabin number within deck
Add CabinMissing Flag

In [44]:
# Data handling is performed in cell 5.


,Deck,Cabin_num,Cabin_num_rel,CabinMissing,Deck_freq
0,B,0.0,0.000000,False,0.089612
1,F,0.0,0.000000,False,0.321408
2,A,0.0,0.000000,False,0.029449
3,A,0.0,0.000000,False,0.029449
4,F,1.0,0.000528,False,0.321408


In [45]:
# Data handling is performed in cell 5.


,Deck,Cabin_num,Cabin_num_rel,CabinMissing,Deck_freq
0,G,3.0,0.000000,False,0.285714
1,F,4.0,0.000000,False,0.337854
2,C,0.0,0.000000,False,0.083002
3,C,1.0,0.002933,False,0.083002
4,F,5.0,0.000530,False,0.337854


### Spending
Create TotalSpent
Create log1p TotalSpent
Create spend properties for Spa, VRDeck, RoomService, FoodCourt, ShoppingMall
Create ZeroSpend flas
Create HighSpend percentile flag

In [46]:
# Data handling is performed in cell 5.


,TotalSpent,LogTotalSpent,ZeroSpend,HighSpend
0,0.0,0.000000,1,0
1,736.0,6.602588,0,0
2,10383.0,9.248021,0,1
3,5176.0,8.551981,0,0
4,1091.0,6.995766,0,0


In [47]:
# Data handling is performed in cell 5.


,TotalSpent,LogTotalSpent,ZeroSpend,HighSpend
0,0.0,0.000000,1,0
1,2832.0,7.949091,0,0
2,0.0,0.000000,1,0
3,7418.0,8.911800,0,1
4,645.0,6.470800,0,0


### CryoSleep
if missing and TotalSpend == 0 set True
if missing and TotalSpend != 0 set False
add inconsistency flag (CryoSleep True but spend > 0) 

In [48]:
# Data handling is performed in cell 5.


,CryoSleep,CryoSleep_inconsistent
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [49]:
# Data handling is performed in cell 5.


,CryoSleep,CryoSleep_inconsistent
0,1,0
1,0,0
2,1,0
3,0,0
4,0,0


### Group features
Group size
Group mean TotalSpend (out of fold)
Group CryoSleep ratio (out of fold)
Individual spend mins group mean
Flag if group all zero spend

In [50]:
# Data handling is performed in cell 5.


,Group_mean_spend,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean
0,1453.141645,0,0.360800,-1453.141645
1,1444.678648,0,0.359454,-708.678648
2,1424.740725,0,0.361806,8958.259275
3,1424.740725,0,0.361806,3751.259275
4,1424.740725,0,0.361806,-333.740725


In [51]:
# Data handling is performed in cell 5.


,Group_mean_spend,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean
0,1440.866329,1,0.360635,-1440.866329
1,1440.866329,1,0.360635,1391.133671
2,1440.866329,1,0.360635,-1440.866329
3,1440.866329,1,0.360635,5977.133671
4,1440.866329,1,0.360635,-795.866329


### Name
Extract last name
Compute surname frequency
Compute out of fold target encoding for surname

In [52]:
# Data handling is performed in cell 5.


,Name,Surname,Surname_freq
0,Maham Ofracculy,Ofracculy,1
1,Juanna Vines,Vines,4
2,Altark Susent,Susent,6
3,Solam Susent,Susent,6
4,Willy Santantines,Santantines,6


In [53]:
# Data handling is performed in cell 5.


,Name,Surname,Surname_freq
0,Nelly Carsoning,Carsoning,4
1,Lerome Peckers,Peckers,1
2,Sabih Unhearfus,Unhearfus,1
3,Meratz Caltilter,Caltilter,1
4,Brence Harperez,Harperez,3


### Categorical Encoding
Use K fold target encoding with smoothing and noise for:
    HomePlanet
    Destination
    Deck
    Surname
    Group ID
Ensure no leaking: encoding must be done only on training folds

In [54]:
# Data handling is performed in cell 5.


,HomePlanet_te,Destination_te,Deck_te,Surname_te,Group_ID_te
0,0.655610,0.470533,0.722140,0.503624,0.476034
1,0.425299,0.473258,0.445917,0.834541,0.477605
2,0.657524,0.469041,0.497702,0.875906,0.474391
3,0.657524,0.469041,0.497702,0.875906,0.559157
4,0.420591,0.469041,0.440913,0.417271,0.474391


Missing values:
VIP missing -> False
Age impute using median per HomePlanet
Cabin missing treated as category

In [55]:
# Data handling is performed in cell 5.


C:\Users\sbrad\AppData\Local\Temp\ipykernel_18320\403975049.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train['VIP'] = X_train['VIP'].fillna(False).astype(int)


,VIP,Age,Deck,Side
0,0,39.0,B,P
1,0,24.0,F,S
2,1,58.0,A,S
3,0,33.0,A,S
4,0,16.0,F,S


In [56]:
# Data handling is performed in cell 5.


C:\Users\sbrad\AppData\Local\Temp\ipykernel_18320\2973202822.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_submission_test['VIP'] = X_submission_test['VIP'].fillna(False).astype(int)


,VIP,Age,Deck,Side
0,0,27.0,G,S
1,0,19.0,F,S
2,0,31.0,C,S
3,0,38.0,C,S
4,0,20.0,F,S


## Modeling

In [57]:
# Build feature list: drop identifiers and raw text ids
drop_cols = [c for c in ['PassengerId', 'Name', 'Cabin'] if c in X_train.columns]
features = [c for c in X_train.columns if c not in drop_cols + ['Transported', 'target']]
print('feature count:', len(features))
features[:50]


feature count: 33


['HomePlanet',
 'CryoSleep',
 'Destination',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'Group_num',
 'Group_ID',
 'Group_size',
 'Deck',
 'Cabin_num',
 'Side',
 'CabinMissing',
 'Cabin_num_rel',
 'TotalSpent',
 'LogTotalSpent',
 'ZeroSpend',
 'HighSpend',
 'CryoSleep_inconsistent',
 'Group_mean_spend',
 'Group_zero_spend',
 'Group_cryo_ratio',
 'Spend_minus_group_mean',
 'Surname',
 'Surname_freq',
 'HomePlanet_te',
 'Destination_te',
 'Surname_te',
 'Group_ID_te']

In [58]:
# Keep feature order aligned with training columns
drop_cols = [c for c in ['PassengerId', 'Name', 'Cabin'] if c in X_submission_test.columns]
submission_features = [c for c in X_submission_test.columns if c not in drop_cols + ['Transported', 'target']]
features = [c for c in features if c in submission_features]
print('feature count:', len(features))
features[:50]


feature count: 29


['HomePlanet',
 'CryoSleep',
 'Destination',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'Group_num',
 'Group_ID',
 'Group_size',
 'Deck',
 'Cabin_num',
 'Side',
 'CabinMissing',
 'Cabin_num_rel',
 'TotalSpent',
 'LogTotalSpent',
 'ZeroSpend',
 'HighSpend',
 'CryoSleep_inconsistent',
 'Group_mean_spend',
 'Group_zero_spend',
 'Group_cryo_ratio',
 'Spend_minus_group_mean',
 'Surname',
 'Surname_freq']

Use XGBClassifier with:
    tree_method = hist
    eval_metric = logloss
    learning_rate small (0.02 - 0.05)
    max_depth (4-8)
    subsample and colsample_bytree tuned
    strong regularization
    early stopping on validation fold

In [59]:
# XGBoost CV example using GroupKFold (use xgb.train with DMatrix for reliable early stopping)
from sklearn.metrics import roc_auc_score
params = dict(tree_method='hist', learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=2.0)
oof = np.zeros(len(X_train))
fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train['Group_num'] if 'Group_num' in X_train.columns else None)):
    X_tr, X_val = X_train[features].iloc[tr_idx].copy(), X_train[features].iloc[val_idx].copy()
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    # encode object/category columns to integer codes using train mapping (avoid DMatrix categorical issues)
    cat_cols = list(X_tr.select_dtypes(include=['object','category']).columns)
    for col in cat_cols:
        train_cats = pd.Categorical(X_tr[col].astype(str))
        mapping = {cat: i for i, cat in enumerate(train_cats.categories)}
        X_tr[col] = X_tr[col].map(mapping).astype(float).fillna(-1)
        X_val[col] = X_val[col].map(mapping).astype(float).fillna(-1)
    # fallback: factorize any remaining object columns
    for col in X_tr.select_dtypes(include=['object']).columns:
        X_tr[col], _ = pd.factorize(X_tr[col])
        X_val[col], _ = pd.factorize(X_val[col])
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    params_train = params.copy()
    params_train.update({'objective':'binary:logistic','eval_metric':'logloss'})
    bst = xgb.train(params_train, dtrain, num_boost_round=1000, evals=[(dval,'valid')], early_stopping_rounds=50, verbose_eval=False)
    # get best ntree limit if available
    best_ntree = getattr(bst, 'best_ntree_limit', None)
    if best_ntree is None:
        best_ntree = getattr(bst, 'best_iteration', None)
    if best_ntree:
        preds = bst.predict(dval, iteration_range=(0, int(best_ntree)))
    else:
        preds = bst.predict(dval)
    oof[val_idx] = preds
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    fold_scores.append(fold_auc)
    print(f'Fold {fold} AUC:', fold_auc)
print('CV mean AUC:', np.mean(fold_scores), 'std:', np.std(fold_scores))

Fold 0 AUC: 0.8904791010141901
Fold 1 AUC: 0.8909511778944723
Fold 2 AUC: 0.9026007375462951
Fold 3 AUC: 0.8988280417078884
Fold 4 AUC: 0.8904152140048959
CV mean AUC: 0.8946548544335483 std: 0.005092764238756722


Train with:
    5 fold GroupKFold
    multiple random seeds (5-10)
    Store out of fold predictions

In [ ]:
# Example: average across multiple seeds using xgb.train for reliable early stopping
from sklearn.metrics import roc_auc_score
seeds = [42, 7, 2021, 345,21321 ,321]
oof_all = np.zeros((len(X_train), len(seeds)))
for i, s in enumerate(seeds):
    params_train = params.copy()
    params_train.update({'objective':'binary:logistic', 'eval_metric':'logloss', 'random_state': s})
    oof_tmp = np.zeros(len(X_train))
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train['Group_num'] if 'Group_num' in X_train.columns else None)):
        X_tr, X_val = X_train[features].iloc[tr_idx].copy(), X_train[features].iloc[val_idx].copy()
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        # convert object dtype columns to pandas 'category' so XGBoost can handle them as categorical
        # encode object/category columns to integer codes using train mapping (avoid DMatrix categorical issues)
        cat_cols = list(X_tr.select_dtypes(include=['object','category']).columns)
        for col in cat_cols:
            train_cats = pd.Categorical(X_tr[col].astype(str))
            mapping = {cat: i for i, cat in enumerate(train_cats.categories)}
            X_tr[col] = X_tr[col].map(mapping).astype(float).fillna(-1)
            X_val[col] = X_val[col].map(mapping).astype(float).fillna(-1)
        # fallback: factorize any remaining object columns
        for col in X_tr.select_dtypes(include=['object']).columns:
            X_tr[col], _ = pd.factorize(X_tr[col])
            X_val[col], _ = pd.factorize(X_val[col])
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        bst = xgb.train(params_train, dtrain, num_boost_round=1000, evals=[(dval, 'valid')], early_stopping_rounds=30, verbose_eval=False)
        best_ntree = getattr(bst, 'best_ntree_limit', None) or getattr(bst, 'best_iteration', None)
        if best_ntree:
            preds = bst.predict(dval, iteration_range=(0, int(best_ntree)))
        else:
            preds = bst.predict(dval)
        oof_tmp[val_idx] = preds
    oof_all[:, i] = oof_tmp

# average across seeds
oof_avg = oof_all.mean(axis=1)
print('Seeds OOF AUC:', roc_auc_score(y_train, oof_avg))

## Ensembling

In [ ]:
# Ensemble: simple average across seed OOFs (oof_all produced earlier)
try:
    ensemble_oof = oof_all.mean(axis=1)
    print('Ensemble OOF AUC:', roc_auc_score(y_train, ensemble_oof))
except NameError:
    print('oof_all not found; run CV cells first')

Ensemble OOF AUC: 0.8934698775664904


Average predictions across:
    All folds
    All seeds
    Optimize seed weights using CV

## Output

Print fold scores
Print overall CV mean and std
roc curves for all

In [ ]:
training_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean,Surname,Surname_freq,HomePlanet_te,Destination_te,Deck_te,Surname_te,Group_ID_te
0,0001_01,Europa,0,B/0/P,TRAPPIST-1e,39.0,0,0.0,0.0,0.0,...,1,0.0,0.0,Ofracculy,1,0.655610,0.470533,0.722140,0.503624,0.476034
1,0002_01,Earth,0,F/0/S,TRAPPIST-1e,24.0,0,109.0,9.0,25.0,...,0,0.0,0.0,Vines,4,0.425299,0.473258,0.445917,0.834541,0.477605
2,0003_01,Europa,0,A/0/S,TRAPPIST-1e,58.0,1,43.0,3576.0,0.0,...,0,0.0,2603.5,Susent,6,0.657524,0.469041,0.497702,0.875906,0.474391
3,0003_02,Europa,0,A/0/S,TRAPPIST-1e,33.0,0,0.0,1283.0,371.0,...,0,0.0,-2603.5,Susent,6,0.657524,0.469041,0.497702,0.875906,0.559157
4,0004_01,Earth,0,F/1/S,TRAPPIST-1e,16.0,0,303.0,70.0,151.0,...,0,0.0,0.0,Santantines,6,0.420591,0.469041,0.440913,0.417271,0.474391


In [ ]:
test_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,LogTotalSpent,ZeroSpend,HighSpend,CryoSleep_inconsistent,Group_mean_spend,Group_zero_spend,Group_cryo_ratio,Spend_minus_group_mean,Surname,Surname_freq
0,0013_01,Earth,1,G/3/S,TRAPPIST-1e,27.0,0,0.0,0.0,0.0,...,0.000000,1,0,0,0.0,1,1.0,0.0,Carsoning,4
1,0018_01,Earth,0,F/4/S,TRAPPIST-1e,19.0,0,0.0,9.0,0.0,...,7.949091,0,0,0,2832.0,0,0.0,0.0,Peckers,1
2,0019_01,Europa,1,C/0/S,55 Cancri e,31.0,0,0.0,0.0,0.0,...,0.000000,1,0,0,0.0,1,1.0,0.0,Unhearfus,1
3,0021_01,Europa,0,C/1/S,TRAPPIST-1e,38.0,0,0.0,6652.0,0.0,...,8.911800,0,1,0,7418.0,0,0.0,0.0,Caltilter,1
4,0023_01,Earth,0,F/5/S,TRAPPIST-1e,20.0,0,10.0,0.0,635.0,...,6.470800,0,0,0,645.0,0,0.0,0.0,Harperez,3


In [ ]:
# before loop
test_preds_seeds = np.zeros((len(X_submission_test), len(seeds)))

for i, s in enumerate(seeds):
    params_train = params.copy()
    params_train.update({'objective':'binary:logistic', 'eval_metric':'logloss', 'random_state': s})
    oof_tmp = np.zeros(len(X_train))
    test_preds_folds = np.zeros((len(X_submission_test), gkf.get_n_splits()))
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=X_train['Group_num'])):
        X_tr = X_train[features].iloc[tr_idx].copy()
        X_val = X_train[features].iloc[val_idx].copy()
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        # create mapping from this fold's train and apply to val and test
        cat_cols = list(X_tr.select_dtypes(include=['object','category']).columns)
        for col in cat_cols:
            train_cats = pd.Categorical(X_tr[col].astype(str))
            mapping = {cat: i for i, cat in enumerate(train_cats.categories)}
            X_tr[col] = X_tr[col].map(mapping).astype(float).fillna(-1)
            X_val[col] = X_val[col].map(mapping).astype(float).fillna(-1)
        # prepare test copy and apply same mapping
        X_test_enc = X_submission_test[features].copy()
        for col in cat_cols:
            X_test_enc[col] = X_test_enc[col].astype(str).map(mapping).astype(float).fillna(-1)

        # fallback factorize if any object remains (rare)
        for col in X_tr.select_dtypes(include=['object']).columns:
            X_tr[col], _ = pd.factorize(X_tr[col])
            X_val[col], _ = pd.factorize(X_val[col])
            X_test_enc[col], _ = pd.factorize(X_test_enc[col])

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        bst = xgb.train(params_train, dtrain, num_boost_round=1000,
                        evals=[(dval, 'valid')], early_stopping_rounds=30, verbose_eval=False)

        best_ntree = getattr(bst, 'best_ntree_limit', None) or getattr(bst, 'best_iteration', None)
        if best_ntree:
            preds = bst.predict(dval, iteration_range=(0, int(best_ntree)))
            test_pred = bst.predict(xgb.DMatrix(X_test_enc), iteration_range=(0, int(best_ntree)))
        else:
            preds = bst.predict(dval)
            test_pred = bst.predict(xgb.DMatrix(X_test_enc))

        oof_tmp[val_idx] = preds
        test_preds_folds[:, fold] = test_pred

    oof_all[:, i] = oof_tmp
    # average across folds for this seed and store
    test_preds_seeds[:, i] = test_preds_folds.mean(axis=1)

# final test prediction: average across seeds
final_test_preds = test_preds_seeds.mean(axis=1)
# boolean label
submission = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Transported': final_test_preds > 0.5})
submission.to_csv('submission.csv', index=False)